# RL training for LLM agents — 5. Calc-X end to end

### One training step driven by hand through Agent Lightning, then a real training run

| # | Notebook | What it builds | Sections |
|---|---|---|---|
| 1 | [GRPO from scratch](1_grpo_from_scratch.ipynb) | an RL trainer in plain PyTorch, switched to verl's algorithm functions once they're checked against ours, then a real inference engine (vLLM) and weight sync | §0–§2 |
| 2 | [From our trainer to verl](2_from_trainer_to_verl.ipynb) | the same trainer, rebuilt from verl's pieces: `DataProto`, Ray actors, worker groups; then plain verl | §3, §5–§6 |
| 3 | [Training real agents](3_training_real_agents.ipynb) | multi-turn tool use, recording a real agent at the HTTP boundary, Agent Lightning, and verl + Agent Lightning end to end | §7–§10 |
| 4 | [Rewards: exact checks and LLM judges](4_rewards_and_judges.ipynb) | what a reward measures, and an LLM judge next to an exact check | own numbering |
| 5 | [Calc-X end to end](5_calc_x_end_to_end.ipynb) | one training step driven by hand through Agent Lightning, then a real training run | own numbering |

Notebooks 1–3 share one set of section numbers, so a reference like "§1.5" or "§8.1" always points to the same place, whichever notebook it appears in. Each notebook runs on its own: code from earlier notebooks is imported from `tutorial/lib/`, where it is copied verbatim.

**Where we are.** Notebooks 1–3 built every piece of an agent RL trainer ourselves and swapped each one for verl's or Agent Lightning's, checking that nothing changed. Notebook 4 looked at the reward. This notebook is the practitioner's view: the real stack, on the full Calc-X task, used the way we'd use it for real.

We play the trainer, one HTTP call at a time: register the model, create a rollout, act as the agent, report a reward, and read back exactly what the Gateway recorded. Then we let the Controller run the real agent in groups, turn the traces into a verl batch, compute advantages the way Agent Lightning does, look at the config, and launch a real training run.

| § | Topic | Key idea |
|---|---|---|
| 0 | The big picture | who does what (verl vs. Agent Lightning), as messages between processes |
| 1 | The task | Calc-X data, and the reward (notebook 4 looks at it in depth) |
| 2 | Rollout engine + Gateway | start vLLM and the Agent Lightning Gateway |
| 3 | Be the agent ourselves | do one rollout by hand and see exactly what is recorded |
| 4 | Real agent rollouts | run the Calc-X agent many times per question (a GRPO *group*) |
| 5 | Traces → `DataProto` | Agent Lightning's `RolloutAdapter`, and where the reward goes |
| 6 | Advantages | per rollout, zero-signal groups, and per-rollout loss normalisation |
| 7 | How verl does it at scale | the training step inside the trainer, and the config tree |
| 8 | A short real training run | launch the actual trainer and read its metrics |

**What we need to know already.** Notebooks 1–3: this one doesn't re-explain log-probs, advantages, PPO, `DataProto` or the gateway idea. Where a topic was covered, we point back to it ("notebook 1, §1.5").

**Hardware:** one GPU with about 80 GB of memory. Background services (vLLM, Gateway, Controller) are started *from* the notebook and logged to `/tmp/agl-tutorial/`.

## 0. The big picture

### RL for LLMs

(Notebook 1 explains this loop in detail; here it is for reference.) In supervised fine-tuning (SFT), we feed the model the *correct* answer and adjust the weights to make each of its tokens more likely. In RL we **don't need** demonstrations to imitate, only a way to check an attempt. Instead we let the model **attempt** (generate a response, maybe calling tools over several turns), **score** the attempt with a reward (here: 1 if the final answer is right, else 0), and then **nudge the model** so that tokens from better-than-expected attempts become more likely and tokens from worse-than-expected attempts become less likely. We repeat this for thousands of attempts.

```
            ┌───────────────────────── policy π_θ  (the LLM weights θ) ─────────────────────────┐
            │                                                                                   │
  question ─┼─► ROLLOUT: generate (multi-turn, with tools) ──► REWARD: is the answer correct?   │
            │                                                          │                        │
            │        ADVANTAGE: better or worse than the other attempts at the same question?   │
            │                                                          │                        │
            └──◄── UPDATE: raise log π(tokens) when advantage > 0, lower it when advantage < 0 ◄─┘
```

### Who does what in this repo?

* **[verl](https://github.com/volcengine/verl)** is an RL **training engine**. It owns the model weights (the *actor*, sharded with FSDP), runs fast generation through an inference engine (vLLM), computes log-probs, advantages and losses, and coordinates GPUs with **Ray**. Out of the box, verl also generates the rollouts itself, through its own "agent loop".
* **Agent Lightning** replaces **only that rollout stage**, so that an **unmodified, real agent** can produce the training data. The agent (here an AutoGen agent with an MCP calculator tool) simply talks to an **OpenAI-compatible Gateway**, which records every token. Agent Lightning then converts those recordings into verl's batch format and hands them back to verl for everything else.

One training step, and who runs each stage:

| Stage | Plain verl | verl + Agent Lightning |
|---|---|---|
| Load prompts from the dataset | verl | verl |
| **Rollout** (run the agent, sample tokens) | verl's `AgentLoop` (e.g. `ToolAgentLoop`), inside verl's Ray workers | **the agent**, in its own process, launched by the Controller, calling the Gateway |
| Token generation | verl's vLLM replicas | verl's vLLM replicas (the Gateway proxies to them) |
| **Reward** | verl reward function, run on the finished batch | reported by the agent as an event |
| Trajectories → `DataProto` | verl agent loop | Agent Lightning `RolloutAdapter` |
| Old/ref log-probs, advantages, loss, optimizer step | verl | verl |
| Push new weights into vLLM, sleep/wake, checkpoints, Ray | verl | verl |

Why replace the rollout stage at all? verl's agent loop is a Python class that works directly on **token IDs**: it applies the chat template, calls `generate(prompt_ids)`, parses tool calls out of the generated tokens and runs tools written as verl `BaseTool` classes. To train an existing agent (AutoGen, LangChain, our own code, MCP tools…) with plain verl, we would have to **rewrite it** as such a class. With Agent Lightning, the agent stays as it is and only its OpenAI base URL changes.

**Why "OpenAI"?** This has nothing to do with OpenAI's models. OpenAI's Chat Completions API (`POST /v1/chat/completions` with a list of messages and optional tools) has become the common protocol for talking to LLMs. vLLM, SGLang, Ollama and most hosted providers serve it, and agent frameworks (AutoGen, LangChain, the `openai` Python client) all speak it, usually with a configurable `base_url`. So it's the one place where almost any agent can be pointed somewhere else without code changes. Agent Lightning puts its Gateway there: it speaks the same API to the agent and forwards each call to vLLM, which also serves that API.

One training step, as messages between the processes (numbers are the order; the section that shows each by hand is in brackets):

```
 Trainer (verl)                Gateway                 Controller               Agent                 vLLM
        │                         │                         │                     │                     │
        │── 1 register vLLM URL ─►│                         │                     │                     │
        │── 2 create rollouts ───►│                         │                     │                     │
        │                         │◄─ 3 any queued? ────────│                     │                     │
        │                         │                         │── 4 start process ─►│                     │
        │                         │◄─ 5 chat completion ──────────────────────────│                     │
        │                         │── forward + ask for tokens ────────────────────────────────────────►│
        │                         │◄─ text, token IDs, log-probs ───────────────────────────────────────│
        │                         │── text only ─────────────────────────────────►│                     │
        │                         │◄─ 6 reward event ─────────────────────────────│                     │
        │                         │                         │◄─ exit 0 ───────────│                     │
        │                         │◄─ 7 mark succeeded ─────│                     │                     │
        │◄─ 8 tokens + rewards ───│                         │                     │                     │
        │ 9 advantages, update    │                         │                     │                     │
        │── 10 push new weights ───────────────────────────────────────────────────────────────────────►│
```

The **Agent** is a program that *uses* the model: it builds prompts, runs tools, decides when to stop and computes the reward. **vLLM** serves the model being trained: it turns messages into sampled tokens and knows nothing about tools or rewards. Training changes only the weights behind vLLM (step 10); the agent's code never changes.

1–2: the trainer tells the Gateway where the policy is served and which questions to attempt (§3.1, §3.2). 3–4: the Controller polls for queued rollouts and starts one agent process each (§3.3, §4.2). 5: the agent makes ordinary chat calls, as many as it likes. The Gateway forwards them to vLLM, keeps the token IDs and log-probs, and returns only the text the agent expects (§3.4). 6–7: the agent reports its reward, and the rollout is marked finished (§3.5). 8–10: the trainer collects the recorded tokens and rewards and does everything else itself (§3.6–§7).

The three Agent Lightning components:

* **API Gateway** (`agentlightning/server`) stores rollouts and events, and proxies the agent's LLM calls to vLLM while capturing token IDs and log-probs.
* **Rollout Controller** (`agentlightning/controller`) watches for queued rollouts and launches one agent process per rollout.
* **Trainer** (`agentlightning/verl`) is a subclass of verl's `RayPPOTrainer`. Its `_train_step` replaces verl's `generate_sequences` call with `_rollout()` (create rollouts, wait for them, build the batch). After that, it calls verl's own functions for the update.

In this notebook **we play the trainer**, one function call at a time.

### 0.1 Setup

This cell locates the repository, checks that we are on the right kernel, and defines the ports and model we will use. The ports deliberately differ from `examples/calc_x/run_local.sh` (8181) so they don't clash with anything else we may have running.

In [ ]:
import os
import sys
import time
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "calc_x").is_dir())
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "examples" / "calc_x"))  # for `eval_utils`, used by the Calc-X agent

try:
    import verl  # noqa: F401
except ImportError as exc:
    raise RuntimeError("verl is not importable: select the 'Python (agl-rl-tutorials .venv)' kernel.") from exc

VENV_BIN = Path(sys.executable).parent

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # the policy we train (same as the Calc-X example)
VLLM_PORT = 8100                       # standalone vLLM (stands in for verl's rollout engine)
AGL_PORT = 8282                        # Agent Lightning Gateway
AGL_KEY = "tutorial"                   # shared secret between trainer, Gateway and Controller
VLLM_GPU_UTIL = 0.45                   # fraction of GPU memory vLLM may use (weights + KV cache + CUDA graphs)

VLLM_URL = f"http://127.0.0.1:{VLLM_PORT}/v1"
AGL_URL = f"http://127.0.0.1:{AGL_PORT}"
LOG_DIR = Path("/tmp/agl-tutorial")
LOG_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda"

import torch
import verl as _verl

print("repo      :", REPO)
print("python    :", sys.executable)
print("verl      :", _verl.__version__, "| torch:", torch.__version__)
print("GPU       :", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB)")

The next cell is plumbing: small helpers to start and stop background services and wait until they are healthy. Each service gets its own process group, so stopping it also stops its children (vLLM spawns an engine process, and the Controller spawns agents). Everything is stopped automatically when the kernel shuts down, and §9 stops it all explicitly.

In [ ]:
import atexit
import signal
import socket
import subprocess

import httpx

SERVICES: dict[str, subprocess.Popen] = {}


def _service_env(extra: dict[str, str] | None = None) -> dict[str, str]:
    env = dict(os.environ)
    env["PATH"] = f"{VENV_BIN}:{env.get('PATH', '')}"  # vLLM's JIT needs `ninja`; the agent runs `python -m mcp_server_calculator`
    env["PYTHONPATH"] = f"{REPO}:{env['PYTHONPATH']}" if env.get("PYTHONPATH") else str(REPO)
    env.update(extra or {})
    return env


def port_in_use(port: int) -> bool:
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0


def log_tail(name: str, n: int = 20) -> str:
    path = LOG_DIR / f"{name}.log"
    return "\n".join(path.read_text(errors="replace").splitlines()[-n:]) if path.exists() else "(no log)"


def start_service(name: str, cmd: list[str], *, cwd: Path = REPO, env: dict[str, str] | None = None) -> subprocess.Popen:
    stop_service(name)
    log_path = LOG_DIR / f"{name}.log"
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd=cwd, env=_service_env(env), stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    SERVICES[name] = proc
    print(f"started {name} (pid {proc.pid}), log: {log_path}")
    return proc


def stop_service(name: str, timeout: float = 30) -> None:
    proc = SERVICES.pop(name, None)
    if proc is None or proc.poll() is not None:
        return
    os.killpg(proc.pid, signal.SIGTERM)
    try:
        proc.wait(timeout)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
    print(f"stopped {name}")


def stop_all() -> None:
    for name in reversed(list(SERVICES)):
        stop_service(name)


atexit.register(stop_all)


def wait_until_healthy(name: str, url: str, timeout: float = 600) -> None:
    start = time.time()
    while time.time() - start < timeout:
        proc = SERVICES.get(name)
        if proc is not None and proc.poll() is not None:
            raise RuntimeError(f"{name} exited with code {proc.returncode}. Last log lines:\n{log_tail(name, 30)}")
        try:
            if httpx.get(url, timeout=2).status_code == 200:
                print(f"{name} is healthy after {time.time() - start:.0f}s")
                return
        except httpx.HTTPError:
            pass
        time.sleep(2)
    raise TimeoutError(f"{name} not healthy after {timeout}s. Last log lines:\n{log_tail(name, 30)}")


print("helpers ready")

## 1. The task: Calc-X, and the reward

Calc-X is a collection of math word problems (arithmetic, algebra, multiple choice). Each row has a `question` and a ground-truth `result`. There is also a `chain` column with a worked solution, but **RL never looks at it**. That is the key difference from SFT: we don't need demonstrations, only a way to *check* answers.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
train_df = pd.read_parquet(REPO / "examples/calc_x/data/train.parquet")
print(f"{len(train_df)} training questions; columns: {list(train_df.columns)}")
print(train_df["source"].value_counts().to_string())
train_df[["question", "result", "source"]].sample(5, random_state=0)

In [ ]:
from pprint import pprint

pprint(train_df.chain[0])

The reward is the agent's own exact check: it extracts the text between `### ANSWER:` and `###` and compares it with `result` using `scalar_are_results_same` (from `examples/calc_x/eval_utils.py`): exact match, multiple-choice letter match, or numeric equality within 1%. [Notebook 4](4_rewards_and_judges.ipynb) looks at what that check really measures, and puts an LLM judge next to it. A few examples:

In [ ]:
from eval_utils import scalar_are_results_same

for pred, truth in [("42", "42"), ("1/2", "0.5"), ("0.501", "0.5"), ("(B)", "B"), ("105", "A"), ("about 42", "42")]:
    reward = 1.0 if scalar_are_results_same(pred, truth, 1e-2) else 0.0
    print(f"predicted {pred!r:>11}  vs  truth {truth!r:>5}  ->  reward {reward}")

## 2. The tokenizer, and the rollout engine

The policy is `Qwen2.5-1.5B-Instruct`. Notebook 1 (§1.2–§1.3) covered what a policy is and how log-probs are computed from it; here we only need its tokenizer, to read the token IDs the Gateway records.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
messages = [{"role": "user", "content": "What is 2+3? Reply with just the number."}]
prompt_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

print(f"{len(prompt_ids)} prompt tokens. Decoded:\n")
print(tokenizer.decode(prompt_ids))
print("first 12 token ids:", prompt_ids[:12])

Generating thousands of multi-turn rollouts with `model.generate()` would be far too slow, so RL frameworks use a dedicated inference engine. verl supports vLLM and SGLang.

**How it works in real verl training:** verl runs its heavy work in separate worker processes, one per GPU, which it starts and coordinates with [Ray](https://docs.ray.io) ([notebook 2](2_from_trainer_to_verl.ipynb), §5–§6, builds it by hand and then with verl). Each of those GPU workers holds both the trainable weights and a vLLM engine on the same GPU. verl calls this the **hybrid engine**. Each step, vLLM is woken up to generate, put to sleep (KV cache freed) during the update, and then receives the freshly updated weights. Here we skip all of that and start a standalone vLLM server with the same model, which is exactly what the trainer would register with the Gateway at step 0.

Notable flags:
* `--enable-auto-tool-choice --tool-call-parser hermes`: Qwen writes tool calls as `<tool_call>{...}</tool_call>` text, and this parser turns them into OpenAI `tool_calls` objects for the agent framework.
* `--generation-config vllm`: ignore the sampling defaults in the model's `generation_config.json` (`top_k=20, top_p=0.8, repetition_penalty=1.1` for Qwen2.5-Instruct). Otherwise vLLM applies them to every request that doesn't set them, and the Gateway only sets `temperature`, so rollouts would be sampled from a truncated distribution instead of the policy itself. verl overrides the same defaults on the vLLM servers it starts.
* `--gpu-memory-utilization`: the slice of GPU memory vLLM pre-allocates (weights + KV cache + CUDA graphs).

This takes about 30-90 s, longer on the very first run while kernels compile.

In [ ]:
if port_in_use(VLLM_PORT):
    print(f"port {VLLM_PORT} already in use; assuming vLLM from a previous run is still up and reusing it")
else:
    start_service("vllm", [
        str(VENV_BIN / "vllm"), "serve", MODEL,
        "--host", "127.0.0.1", "--port", str(VLLM_PORT),
        "--gpu-memory-utilization", str(VLLM_GPU_UTIL),
        "--max-model-len", "8192",
        "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
        "--generation-config", "vllm",
    ])
wait_until_healthy("vllm", f"http://127.0.0.1:{VLLM_PORT}/health")

Next, the **Gateway** (`agl-server`). Why not point the agent straight at vLLM? The trainer needs three things that a plain OpenAI endpoint doesn't give it:

* **The exact tokens.** Training needs the token IDs the model actually sampled, and their log-probs. The agent framework only ever sees *text*, and re-tokenizing text afterwards doesn't always give back the same tokens. The Gateway asks vLLM for token IDs and log-probs on every call and stores them (§3.6).
* **Which rollout each call belongs to.** Many agents run at once, each making several calls. The rollout ID in the proxy URL lets the Gateway attach every call to its rollout, so the trainer can group calls into episodes and episodes into GRPO groups.
* **Control the agent doesn't know about.** The Gateway enforces training settings (sampling temperature, token and log-prob capture) and routes to whichever vLLM servers currently hold the latest weights. The agent needs no RL code, only a base URL.

It is also where the trainer, Controller and agents share state: rollout status, rewards and other events. It is a small FastAPI app with an in-memory store (`agentlightning/server/store.py`: three Python dicts!) and four groups of routes:

| Route | Used by | Purpose |
|---|---|---|
| `POST/DELETE /api/models` | trainer | register the vLLM endpoint(s) currently serving the policy |
| `POST/GET/PATCH /api/rollouts` | trainer, Controller | create rollouts, list queued ones, move them through `queuing → running → succeeded/failed` |
| `POST/GET /api/rollouts/{id}/…/events` | agent, trainer | append events (reward, custom), read them back (raw or `format=triplet`) |
| `POST /proxy/rollout/{id}/attempt/{a}/mode/{train\|val}/openai/v1/chat/completions` | agent | OpenAI-compatible proxy; records each call as a `model_request` event |

`default_proxy.model_name` must match the model name vLLM serves.

In [ ]:
if port_in_use(AGL_PORT):
    print(f"port {AGL_PORT} already in use; assuming the Gateway from a previous run is still up and reusing it")
else:
    start_service("gateway", [
        sys.executable, "-m", "agentlightning.server",
        "host=127.0.0.1", f"port={AGL_PORT}", f"key={AGL_KEY}",
        f"default_proxy.model_name={MODEL}",
        f"hydra.run.dir={LOG_DIR}/hydra/gateway", "hydra.output_subdir=null",
    ])
wait_until_healthy("gateway", f"{AGL_URL}/healthz")

agl = httpx.Client(base_url=AGL_URL, headers={"Authorization": f"Bearer {AGL_KEY}"}, timeout=120)
print("Gateway /healthz ->", agl.get("/healthz").json())

## 3. Be the agent: one rollout, by hand

Before running the real agent, let's do everything the trainer, Controller and agent would do, ourselves, with plain HTTP calls. That makes every piece of state visible.

### 3.1 Register the model endpoint (trainer's job)

At the start of every rollout phase the trainer calls `delete_model()` and then `register_model(server_addresses)` (`agentlightning/verl/agl_rollout_manager.py`), telling the Gateway where the *current* policy is being served.

In [ ]:
agl.delete("/api/models")
print(agl.post("/api/models", json=[{"model": MODEL, "endpoint": VLLM_URL}]).json())

### 3.2 Create a rollout (trainer's job)

A **rollout** is one execution of an agent on one input. Its `input` can be any JSON; for Calc-X it's a dataset row. The new rollout starts in state `queuing`. (In §4 the Controller will pick queued rollouts up automatically; it isn't running yet, so this one will wait for us.)

In [ ]:
import json

manual_input = {"question": "What is 17 * 23? Then add 9.", "result": "400"}
rollout = agl.post("/api/rollouts", json=[{"input": manual_input, "is_train": True}]).json()[0]
rid = rollout["rollout_id"]
print(json.dumps(rollout, indent=2))

### 3.3 Start it (Controller's job)

The Controller moves the rollout to `running` when it launches the agent process. The Gateway enforces a small state machine (`VALID_TRANSITIONS` in `agentlightning/schemas.py`).

In [ ]:
r = agl.patch(f"/api/rollouts/{rid}", json={"status": {"state": "running", "last_attempt_id": "0"}})
print(r.status_code, r.json()["status"]["state"])

### 3.4 Talk to the model through the proxy (agent's job)

The agent gets a **rollout-specific base URL** and uses it like any OpenAI endpoint. Because the rollout ID is part of the URL, the Gateway knows which rollout each call belongs to. The agent doesn't need any RL code at all.

We do a two-turn conversation. The second user message plays the role of an *observation from the environment* (in the real agent this is a calculator tool result).

Behind the scenes, in `train` mode the Gateway rewrites each request (`ProxyRouter.prepare_body` in `agentlightning/server/proxy.py`):
* `temperature = 1.0` (together with `--generation-config vllm` from §2), so we **sample from the actual policy** distribution. This keeps the data on-policy and makes the recorded log-probs match the model's real softmax.
* `return_token_ids = true` and `logprobs = true`, so vLLM returns the exact prompt/response token IDs and per-token log-probs, which the Gateway stores.

In [ ]:
from openai import OpenAI

proxy_base_url = f"{AGL_URL}/proxy/rollout/{rid}/attempt/0/mode/train/openai/v1"
llm = OpenAI(base_url=proxy_base_url, api_key=AGL_KEY)
print("agent's OpenAI base URL:", proxy_base_url, "\n")

FORMAT = "Keep it short. Your last line must be exactly: ### ANSWER: <number> ###"
convo = [{"role": "user", "content": f"What is 17 * 23? {FORMAT}"}]
reply1 = llm.chat.completions.create(model="whatever-the-agent-thinks", messages=convo, max_tokens=200).choices[0].message.content
print("TURN 1 ->", reply1, "\n")

convo += [
    {"role": "assistant", "content": reply1},
    {"role": "user", "content": f"Now add 9 to your result. {FORMAT}"},  # the "observation"
]
reply2 = llm.chat.completions.create(model="whatever-the-agent-thinks", messages=convo, max_tokens=200).choices[0].message.content
print("TURN 2 ->", reply2)

### 3.5 Report the reward and finish (agent's and Controller's job)

The agent computes its own reward (it knows the expected `result` from its input) and posts it as an event. Then the Controller marks the rollout `succeeded` once the process exits with code 0. Terminal states are final: the last request below shows the Gateway rejecting an invalid transition.

In [ ]:
import re


def extract_answer(text: str) -> str:
    match = re.search(r"###\s*ANSWER:\s*(.+?)(\s*###|$)", text)
    return match.group(1).strip() if match else text.strip()


manual_reward = 1.0 if scalar_are_results_same(extract_answer(reply2), manual_input["result"], 1e-2) else 0.0
print(f"final answer {extract_answer(reply2)!r} vs {manual_input['result']!r} -> reward {manual_reward}")

agl.post(f"/api/rollouts/{rid}/attempt/0/events", json={"event_type": "reward", "data": {"value": manual_reward}}).raise_for_status()
print("succeeded:", agl.patch(f"/api/rollouts/{rid}", json={"status": {"state": "succeeded"}}).json()["status"]["state"])
bad = agl.patch(f"/api/rollouts/{rid}", json={"status": {"state": "running"}})
print("try succeeded -> running:", bad.status_code, bad.json()["detail"])

Since the model samples at temperature 1, our results differ from run to run. If the math is right but the reward is 0, look at the format: `### Answer:` or `\[400\]` don't match the `### ANSWER: … ###` pattern. That is the "the model optimizes exactly what we check" point from notebook 4 in action. Early in RL training, a large share of the improvement often comes from the model simply learning the expected answer format.

### 3.6 What did the Gateway record?

Each LLM call became a `model_request` event holding the full request and response (several KB). The trainer reads the compact **triplet** view (`format=triplet`): for each call, the `prompt_token_ids`, the `response_token_ids`, and vLLM's `response_log_probs`. Everything downstream is built from these token IDs, **not** from re-tokenizing text. That avoids a whole class of subtle bugs where `decode → encode` doesn't round-trip.

In [ ]:
raw_events = agl.get(f"/api/rollouts/{rid}/events").json()
print("raw events:", [e["event_type"] for e in raw_events])
print("raw model_request keys:", sorted(raw_events[0]["data"].keys()), "\n")

triplets = agl.get(f"/api/rollouts/{rid}/events", params={"format": "triplet"}).json()
calls = [e["data"] for e in triplets if e["event_type"] == "model_request"]
for i, c in enumerate(calls):
    print(f"call {i}: {len(c['prompt_token_ids'])} prompt tokens, {len(c['response_token_ids'])} response tokens")
print("reward event:", [e["data"] for e in triplets if e["event_type"] == "reward"])

print("\nFirst 12 response tokens of call 0, with the log-prob vLLM sampled them at:")
c0 = calls[0]
for tid, lp in list(zip(c0["response_token_ids"], c0["response_log_probs"]))[:12]:
    print(f"   {tid:>7}  {tokenizer.decode([tid])!r:>14}  log p = {lp:+.3f}")

### 3.7 The key structural fact: prefix continuity

Look at how call 1's prompt relates to call 0. If `prompt₁` starts with exactly `prompt₀ + response₀`, then the whole conversation is **one growing token sequence**:

```
[ prompt₀ ][ response₀ ][ observation tokens ][ response₁ ]
            ^ model       ^ environment         ^ model
```

Then the two calls can be trained as **one row**: the model's tokens are trained, and the observation tokens are kept as context but *masked out* of the loss (the model didn't choose them). Agent Lightning's `trajectory` aggregation checks this, token for token.

In [ ]:
p0, r0, p1 = calls[0]["prompt_token_ids"], calls[0]["response_token_ids"], calls[1]["prompt_token_ids"]
continuous = p1[: len(p0) + len(r0)] == p0 + r0
print("prompt1 starts with prompt0 + response0 ?", continuous)
if continuous:
    print("\nObservation tokens inserted between the two model turns (these get loss-masked):")
    print(repr(tokenizer.decode(p1[len(p0) + len(r0):])))

### 3.8 Package it the way the trainer does

Once a rollout is terminal, the trainer's `AglRolloutManager` fetches its triplet events and builds a `CompletedRollout`: a list of `Triplet(prompt, response, reward)` objects with the final reward attached to the last one. We call the same (internal) method so we get the exact object the trainer would get.

In [ ]:
from agentlightning.verl.agl_rollout_manager import AglRolloutManager, EnqueuedRollout

_helper = AglRolloutManager(agl_base_url=AGL_URL, agl_key=AGL_KEY, model=MODEL, step=0)
manual_completed = _helper._build_completed_rollout(
    EnqueuedRollout(data_id="manual-question", rollout_id=rid, step=0, sample_idx_in_step=0, enqueue_time=time.time(), input=manual_input),
    _helper._get_rollout(rid),
)
print("state:", manual_completed.rollout_state, "| final_reward:", manual_completed.final_reward)
for t in manual_completed.triplets:
    print(f"Triplet(prompt={len(t.prompt['token_ids'])} tokens, response={len(t.response['token_ids'])} tokens, reward={t.reward})")

## 4. Real rollouts: the Calc-X agent and the Controller

### 4.1 The agent

Here is the whole agent (`examples/calc_x/calc_agent.py`). It is an ordinary AutoGen `AssistantAgent` with an MCP calculator tool. The only Agent Lightning-specific parts are the environment variables it reads:

| Env var | Set by | Meaning |
|---|---|---|
| `AGL_OPENAI_BASE_URL` | Controller | the rollout-specific proxy URL (like `proxy_base_url` in §3.4) |
| `AGL_EVENT_URL` | Controller | where to POST the reward event |
| `AGL_KEY` | Controller | API key |
| `QUESTION`, `RESULT` | Controller, from `local.env_map` | fields of the rollout's `input` (the dataset row) |

In [ ]:
print((REPO / "examples/calc_x/calc_agent.py").read_text())

### 4.2 The Controller

The Controller (`agentlightning/controller/local_reconciler.py`) is a reconcile loop. Every `poll_interval` seconds it lists `queuing`/`running` rollouts, spawns one Python subprocess per queued rollout (importing the class named in the rollout's `config.local.agent_class` and calling `.run()`), and patches the rollout to `succeeded`/`failed` when the process exits. It also enforces the per-rollout timeout. In Kubernetes mode, the same loop creates one Job per rollout instead.

We start it from `examples/calc_x/` so the agent can `import eval_utils`.

In [ ]:
start_service("controller", [
    sys.executable, "-m", "agentlightning.controller",
    "runner_type=local", f"agl_server.url={AGL_URL}", f"agl_server.key={AGL_KEY}",
    "local_runner.poll_interval=1",
    f"hydra.run.dir={LOG_DIR}/hydra/controller", "hydra.output_subdir=null",
], cwd=REPO / "examples/calc_x")
time.sleep(3)
assert SERVICES["controller"].poll() is None, log_tail("controller")
print("\n".join(l for l in log_tail("controller", 200).splitlines() if "HTTP Request" not in l)[-500:])

### 4.3 Groups of rollouts: what GRPO needs

GRPO (Group Relative Policy Optimization) learns by **comparison**: for each question it samples `n` independent attempts (a *group*) and asks which ones did better than the group's average. Calc-X uses `actor_rollout_ref.rollout.n = 4`.

We pick `NUM_QUESTIONS` questions (a mini version of `data.train_batch_size`, which is 32 in Calc-X) and hand them to `AglRolloutManager.enqueue_and_wait_until_completed`, the exact call the trainer makes in `_rollout()`. `data_dict` has the same shape as what verl's dataloader produces: one numpy object array per dataset column.

This should take well under a minute: each rollout is a separate agent process, and they run in parallel.

In [ ]:
NUM_QUESTIONS = 16
ROLLOUT_N = 4
SEED = 7

picked = train_df.sample(n=NUM_QUESTIONS, random_state=SEED)
data_dict = {col: np.array(picked[col].tolist(), dtype=object) for col in picked.columns}

manager = AglRolloutManager(
    agl_base_url=AGL_URL,
    agl_key=AGL_KEY,
    model=MODEL,
    step=0,
    train_rollout_n=ROLLOUT_N,
    rollout_timeout_seconds=300,
    poll_interval_seconds=3,
    local_agent_class="examples.calc_x.calc_agent.Agent",
    local_env_map={"QUESTION": "input.question", "RESULT": "input.result"},
)
manager.delete_model()
manager.register_model([VLLM_URL])

t0 = time.time()
completed = manager.enqueue_and_wait_until_completed(data_dict, is_train=True)
print(f"\n{len(completed)} rollouts finished in {time.time() - t0:.0f}s")

In [ ]:
summary = pd.DataFrame([
    {
        "q": r.sample_idx_in_step,
        "data_id": r.data_id[:8],
        "rollout_id": r.rollout_id[:8],
        "state": str(r.rollout_state),
        "reward": r.final_reward,
        "model_calls": len(r.triplets or []),
        "response_tokens": sum(len(t.response["token_ids"]) for t in r.triplets or []),
    }
    for r in completed
]).sort_values(["q", "rollout_id"])

per_question = summary.groupby("q").agg(rewards=("reward", list), mean_reward=("reward", "mean"))
per_question["question"] = [picked.iloc[q]["question"][:70] for q in per_question.index]
per_question["truth"] = [picked.iloc[q]["result"] for q in per_question.index]
print(f"mean reward: {summary.reward.mean():.3f}   states: {summary.state.value_counts().to_dict()}")
per_question

Each row above is a GRPO group. Groups with **mixed** rewards (some 0s and some 1s) are where learning happens. Groups where all `n` attempts got the same reward carry **no signal**, as §6 will show.

Let's read one full trajectory, decoded from the recorded token IDs.

In [ ]:
def print_rollout(r, max_prompt_chars: int = 1500) -> None:
    print(f"rollout {r.rollout_id[:8]}  reward={r.final_reward}  calls={len(r.triplets)}  truth={r.input['result']!r}\n")
    for i, t in enumerate(r.triplets):
        prompt_text = tokenizer.decode(t.prompt["token_ids"])
        if i > 0:  # later prompts repeat the conversation; show only the last two messages
            prompt_text = "...<|im_start|>" + "<|im_start|>".join(prompt_text.split("<|im_start|>")[-2:])
        print(f"──── call {i} PROMPT ({len(t.prompt['token_ids'])} tokens) ────")
        print(prompt_text[-max_prompt_chars:])
        print(f"──── call {i} RESPONSE ({len(t.response['token_ids'])} tokens) ────")
        print(tokenizer.decode(t.response["token_ids"]), "\n")


best = max(completed, key=lambda r: (r.final_reward or 0, len(r.triplets or [])))
print_rollout(best)

Things worth noticing in a real agent trace:

1. **The tool schema lives in the system prompt** (`<tools>…</tools>`), and tool calls are just generated text (`<tool_call>{…}</tool_call>`) that vLLM's hermes parser converts to structured calls.
2. **The calculator's output comes back as a `<tool_response>` user message.** Those tokens were not produced by the policy.
3. **The second call's system prompt is different from the first.** AutoGen's `reflect_on_tool_use=True` makes the final "reflection" call *without* tools, so the `<tools>` block disappears. That breaks the prefix continuity from §3.7, which matters in the next section.

## 5. From traces to a verl training batch

### 5.1 `DataProto`: verl's batch container

Everything that flows between verl components is a `verl.DataProto`: tensors in `.batch`, ids and strings in `.non_tensor_batch`, batch-level values in `.meta_info` ([notebook 2](2_from_trainer_to_verl.ipynb), §3).

### 5.2 `RolloutAdapter`: traces → padded tensors

`RolloutAdapter.get_train_data_batch` (`agentlightning/verl/rollout_adapter.py`) turns `CompletedRollout`s into a `DataProto`. Each row is laid out as:

```
input_ids      = [ pad … pad  prompt tokens ][ response tokens  pad … pad ]
                 └── left-padded to max_prompt_length ┘└── right-padded to max_response_length ┘
attention_mask =   0 … 0      1 … 1             1 … 1            0 … 0
response_mask  =                              [ 1 = trained (model's tokens), 0 = observation or pad ]
```

Two aggregation levels (`agentlightning.trace_aggregator.level`):
* **`transition`**: every model call becomes its own row.
* **`trajectory`** (default): consecutive calls are merged into one row *if* the token prefix is continuous; observation tokens go into the response with `response_mask = 0`. If continuity breaks, a new row starts.

Let's first convert our hand-made rollout from §3, which *was* prefix-continuous.

In [ ]:
from agentlightning.verl.rollout_adapter import RolloutAdapter

PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0


def make_adapter(level: str) -> RolloutAdapter:
    # Lengths are Agent Lightning's defaults (agentlightning/verl/config.yaml, trace_aggregator.*).
    return RolloutAdapter(
        max_prompt_length=2048,
        max_response_length=8192,
        device=torch.device("cpu"),
        pad_token_id=PAD_ID,
        reward_fillna_value=0.0,
        trace_aggregator_level=level,
        tokenizer=tokenizer,
    )


def describe(batch) -> None:
    print(f"DataProto with {len(batch)} rows")
    for k, v in batch.batch.items():
        print(f"  batch[{k!r:24}] {str(tuple(v.shape)):14} {v.dtype}")
    for k, v in batch.non_tensor_batch.items():
        print(f"  non_tensor_batch[{k!r}] = {v[:3].tolist()}{' …' if len(v) > 3 else ''}")


manual_batch, manual_metrics = make_adapter("trajectory").get_train_data_batch([manual_completed])
describe(manual_batch)

A colored view of a row makes the masks concrete: grey is the prompt (context only), **green** is response tokens that will be trained, and **orange** is response-side tokens that are context but masked out of the loss.

In [ ]:
import html as _html

from IPython.display import HTML, display

_STYLE = {"prompt": "background:#e8e8e8;color:#555", "train": "background:#b9f0b4;color:#000", "masked": "background:#ffd59e;color:#000"}


def show_row(batch, i: int, title: str = "") -> None:
    P = batch.batch["prompts"].shape[1]
    ids, attn = batch.batch["input_ids"][i].tolist(), batch.batch["attention_mask"][i].tolist()
    resp_mask = batch.batch["response_mask"][i].tolist() if "response_mask" in batch.batch else attn[P:]
    spans, counts = [], {"prompt": 0, "train": 0, "masked": 0}
    for pos, (tid, a) in enumerate(zip(ids, attn)):
        if not a:
            continue  # padding
        kind = "prompt" if pos < P else ("train" if resp_mask[pos - P] else "masked")
        counts[kind] += 1
        if spans and spans[-1][0] == kind:
            spans[-1][1].append(tid)
        else:
            spans.append((kind, [tid]))
    body = "".join(
        f'<span style="{_STYLE[k]}">{_html.escape(tokenizer.decode(t)).replace(chr(10), "↵<br>")}</span>' for k, t in spans
    )
    legend = " ".join(f'<span style="{_STYLE[k]};padding:0 4px">{k}: {n} tokens</span>' for k, n in counts.items())
    display(HTML(
        f"<b>{_html.escape(title)}</b> row {i} | rollout {batch.non_tensor_batch['rollout_id_list'][i][:8]}<br>{legend}"
        f'<div style="font-family:monospace;font-size:12px;max-height:380px;overflow:auto;border:1px solid #ccc;padding:6px;margin-top:4px">{body}</div>'
    ))


show_row(manual_batch, 0, "hand-made 2-turn rollout")

If the prefix check in §3.7 printed `True`, we'll see **one** row: turn 1 (green), the inserted "Now add 9…" message (orange: context only, never trained), then turn 2 (green). If it printed `False`, which can happen when a sampled token doesn't survive decode → re-encode, the adapter safely emits two separate rows instead.

### 5.3 Where does the reward go?

`token_level_scores` has the shape of the response, is zero everywhere, and holds the rollout's reward on the **last** real token of each row. verl's advantage estimators expect this layout. (GRPO only needs the sum, but PPO with a critic would discount it backwards in time.) If a rollout produced several rows, **each** row carries the rollout's final reward. Let's check on the agent rollout we printed in §4.

In [ ]:
best_batch, _ = make_adapter("trajectory").get_train_data_batch([best])
for i in range(len(best_batch)):
    scores = best_batch.batch["token_level_scores"][i]
    nz = scores.nonzero().flatten().tolist()
    last = int(best_batch.batch["attention_mask"][i, best_batch.batch["prompts"].shape[1]:].sum()) - 1
    print(f"row {i}: non-zero token_level_scores at response index {nz} = {scores[nz].tolist()}; last response token index = {last}")
if not best.final_reward:
    print("(this rollout's reward is 0, so the scores are all zero; re-run §4 to get a rollout with reward 1)")

### 5.4 The real agent batch: trajectory vs. transition

Now convert the agent rollouts with both aggregation levels and compare. Because of the tool-schema change described in §4, most Calc-X agent calls **can't** be merged, so the two levels produce nearly the same number of rows. With an agent whose prompts grow append-only, `trajectory` would produce about one row per rollout.

In [ ]:
def continuity_breaks(r) -> int:
    breaks, ctx = 0, None
    for t in r.triplets or []:
        p, resp = t.prompt["token_ids"], t.response["token_ids"]
        if ctx is not None and p[: len(ctx)] != ctx:
            breaks += 1
        ctx = p + resp
    return breaks


train_batch, adapter_metrics = make_adapter("trajectory").get_train_data_batch(completed)
transition_batch, _ = make_adapter("transition").get_train_data_batch(completed)

print(f"{len(completed)} rollouts, {sum(len(r.triplets or []) for r in completed)} model calls in total")
print(f"  trajectory level -> {len(train_batch)} rows (unmerged rollouts: {adapter_metrics['training/n_unmerged_rollouts']})")
print(f"  transition level -> {len(transition_batch)} rows")
print(f"  rollouts with ≥1 prefix-continuity break: {sum(continuity_breaks(r) > 0 for r in completed)}")
print("\nadapter metrics:", {k: v for k, v in adapter_metrics.items() if not k.startswith("timing")})
describe(train_batch)

In [ ]:
row = int(np.argmax([r == best.rollout_id for r in train_batch.non_tensor_batch["rollout_id_list"]]))
show_row(train_batch, row, "Calc-X agent, first row of the rollout printed in §4")

## 6. Advantages: "was this attempt better than its siblings?"

### 6.1 Why not just use the reward?

Notebook 1 (§1.5) explains why GRPO compares each attempt with its group: A_i = (r_i − mean) / (std + ε), the same for every response token of attempt *i*. Here we compute it by hand from the real agent batch, then check it against what Agent Lightning computes.

In [ ]:
rows_df = pd.DataFrame({
    "data_id": train_batch.non_tensor_batch["data_id_list"],
    "rollout_id": train_batch.non_tensor_batch["rollout_id_list"],
    "reward": train_batch.batch["token_level_scores"].float().sum(-1).numpy(),
})
per_rollout = rows_df.drop_duplicates("rollout_id").copy()
grp = per_rollout.groupby("data_id")["reward"]
per_rollout["group_mean"] = grp.transform("mean")
per_rollout["group_std"] = grp.transform("std")  # unbiased (ddof=1), same as torch.std in verl
per_rollout["adv_by_hand"] = (per_rollout.reward - per_rollout.group_mean) / (per_rollout.group_std + 1e-6)
per_rollout.sort_values(["data_id", "reward"]).head(16)

### 6.2 The same thing, the verl way

The trainer does exactly this in the `adv` block of `_train_step` (`agentlightning/verl/trainer.py`):

1. `uid = data_id_list`: the group key;
2. `token_level_rewards = token_level_scores` (when there is no KL penalty in the reward);
3. `compute_rollout_level_advantage(...)`, Agent Lightning's wrapper around verl's `compute_advantage(adv_estimator="grpo")`.

The wrapper exists because one rollout can produce **several rows** (see §5.4). Computing GRPO per *row* would make a rollout with 3 rows count 3 times in its group's mean and std. The wrapper instead computes the advantage once per **rollout** (one representative row each) and broadcasts it to all of that rollout's rows, multiplied by `response_mask`.

In [ ]:
from agentlightning.verl.rollout_level_advantage import compute_rollout_level_advantage

train_batch.non_tensor_batch["uid"] = train_batch.non_tensor_batch["data_id_list"]
train_batch.batch["token_level_rewards"] = train_batch.batch["token_level_scores"]
train_batch, adv_metrics = compute_rollout_level_advantage(
    train_batch, adv_estimator="grpo", gamma=1.0, lam=1.0, num_repeat=ROLLOUT_N, norm_adv_by_std_in_grpo=True, config=None,
)
print(adv_metrics)

adv = train_batch.batch["advantages"].float()
mask = train_batch.batch["response_mask"].bool()
row_adv = torch.stack([adv[i][mask[i]][0] for i in range(len(train_batch))]).numpy()
rows_df["adv_verl"] = row_adv
check = rows_df.merge(per_rollout[["rollout_id", "adv_by_hand"]], on="rollout_id")
print("max |verl - by hand| =", float(np.abs(check.adv_verl - check.adv_by_hand).max()), "(bf16 rounding)")
print("is the advantage constant over each row's trained tokens?", all(torch.allclose(adv[i][mask[i]], adv[i][mask[i]][0]) for i in range(len(train_batch))))
print("is it exactly zero on masked tokens?", bool((adv[~mask] == 0).all()))

### 6.3 Groups with no signal

If all n attempts get the same reward (all right or all wrong), then r − mean = 0 and the advantage is **zero for every token**: the group contributes nothing to the gradient, but still costs rollout and compute time. That's why the trainer logs `training/n_zero_adv_groups`. When aligning rows to mini-batch boundaries, `_train_step` also **drops same-reward groups first** (`_same_reward_uid_indices`). Too-easy or too-hard data, or too small an `n`, shows up as many zero-advantage groups.

In [ ]:
group_stats = per_rollout.groupby("data_id")["reward"].agg(["mean", "std", "count"])
zero = group_stats[group_stats["std"].fillna(0) == 0]
print(f"{len(group_stats)} groups, {len(zero)} with zero advantage "
      f"({(zero['mean'] == 1).sum()} all-correct, {(zero['mean'] == 0).sum()} all-wrong)")
print(f"rows with non-zero advantage (these actually train): {(np.abs(row_adv) > 0).sum()} / {len(row_adv)}")

### 6.4 Per-rollout normalization of the loss

One more Agent Lightning detail. verl's default loss averages over **tokens** (`loss_agg_mode="token-mean"`), so a rollout with more rows or more tokens would get more total weight. Agent Lightning's `per_rollout_mean` loss mode first rescales each row's advantage by `1 / (tokens in its whole rollout × number of rows trained)` (`normalize_advantages_by_rollout`). Then it **sums** the per-token loss. The result: each rollout carries the same total weight no matter how many rows or tokens it produced. The trainer applies this just before `_update_actor`.

In [ ]:
from agentlightning.verl.per_rollout_loss import normalize_advantages_by_rollout

adv_norm = normalize_advantages_by_rollout(
    train_batch.batch["advantages"].float(), train_batch.batch["response_mask"], train_batch.non_tensor_batch["rollout_id_list"],
    num_trained_rows=len(train_batch),
)
mass = pd.DataFrame({
    "rollout_id": train_batch.non_tensor_batch["rollout_id_list"],
    "|A| mass": (adv_norm.abs() * mask).sum(-1).numpy(),
    "trained tokens": mask.sum(-1).numpy(),
}).groupby("rollout_id").sum()
mass["|A_i| (raw GRPO)"] = per_rollout.set_index("rollout_id")["adv_by_hand"].abs()
mass["|A_i| / num_rows"] = mass["|A_i| (raw GRPO)"] / len(train_batch)
print("Each rollout's total |advantage| mass is |A_i| / num_rows, independent of how many tokens or rows it has:")
mass[mass["|A| mass"] > 0].sort_values("trained tokens").head(10)

## The update itself

The next step is the policy update: old log-probs, the PPO-clip loss, the backward pass and the optimizer step. [Notebook 1](1_grpo_from_scratch.ipynb) built every piece of it by hand (§1.3–§2.2), including the check that it worked, and its §2.6 showed why the trainer recomputes the old log-probs instead of trusting vLLM's. Here we let the real trainer do it, in §8; §7 first shows where each piece lives inside it.

## 7. How verl does all of this at scale

### 7.1 The same step, inside `AgentLightningRayPPOTrainer._train_step`

| What we did by hand | Trainer code (`agentlightning/verl/trainer.py`) | verl machinery behind it |
|---|---|---|
| §2 start vLLM | `init_workers()` (once) | vLLM replicas inside Ray workers (`llm_server_manager`) |
| §4 run rollouts | `_rollout()` → `AglRolloutManager.enqueue_and_wait_until_completed` | vLLM replicas woken up; endpoints registered with the Gateway |
| §5 build batch | `RolloutAdapter.get_train_data_batch` | `DataProto` |
| (not shown) | drop `is_drop_mask` rows; align rows to whole mini-batches, dropping same-reward groups first; cap with `max_ppo_update_times` | |
| (not shown) | `checkpoint_manager.sleep_replicas()` | free vLLM's KV cache so the update has GPU memory |
| old log-probs (notebook 1, §1.7) | `_compute_old_log_prob` | actor worker group (FSDP-sharded model) |
| ref log-probs (not used by Calc-X) | `_compute_ref_log_prob` (only with KL) | ref worker group |
| §6 advantages | `compute_rollout_level_advantage` → verl `compute_advantage` | runs on the driver: cheap tensor math |
| §6.4 | `normalize_advantages_by_rollout` | |
| update (notebook 1, §1.7, §2.1) | `_update_actor` → verl `ppo_loss` → `per_rollout_mean` | actor worker group: mini-batches, micro-batches, optimizer |
| weight sync (notebook 1, §2.4) | `checkpoint_manager.update_weights()` | new weights → vLLM |

The rest of `fit()` is bookkeeping: loop over steps and epochs, run `_validate()` every `test_freq` steps (rollouts in `val` mode, greedy decoding, only `val/reward` metrics), `_save_checkpoint()` every `save_freq` steps, and log metrics.

### 7.2 The architecture: a single controller over Ray

verl uses a **single-controller** design: one driver process (the `fit()` loop above) holds the *logic*, while the *heavy compute* happens in **worker groups** of Ray actors, one per GPU. When the driver calls `self.actor_rollout_wg.compute_log_prob(batch)`, verl splits the `DataProto` across the workers, runs the method on every GPU, and gathers the result. [Notebook 2](2_from_trainer_to_verl.ipynb) (§5–§6) builds exactly this, by hand and then with verl's `RayWorkerGroup`.

On a single GPU, everything is **collocated**. The actor (FSDP, with parameters and optimizer offloaded to CPU in Calc-X's config) and the vLLM engine share the device and take turns: vLLM is awake during rollouts and sleeps during the update.

### 7.3 The config: every knob in one tree

verl is configured with Hydra/OmegaConf. Agent Lightning's `agentlightning/verl/config.yaml` extends verl's `ppo_trainer.yaml` and adds an `agentlightning:` section, and `examples/calc_x/train_calc_agent.py:build_config` layers the Calc-X overrides on top. Any key can be overridden from the command line with dotted paths, e.g. `actor_rollout_ref.rollout.n=8`.

In [ ]:
from omegaconf import OmegaConf

from examples.calc_x.train_calc_agent import build_config

config = build_config(model=MODEL, agl_base_url=AGL_URL, agl_key=AGL_KEY)

KEYS = {
    "data.train_batch_size": "questions per step",
    "actor_rollout_ref.rollout.n": "rollouts per question (GRPO group size)",
    "actor_rollout_ref.rollout.temperature": "sampling temperature for training rollouts",
    "actor_rollout_ref.rollout.gpu_memory_utilization": "vLLM's share of GPU memory",
    "actor_rollout_ref.actor.ppo_mini_batch_size": "questions per optimizer step (multiplied by n)",
    "actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu": "rows per forward/backward (gradient accumulation)",
    "actor_rollout_ref.actor.ppo_epochs": "passes over each batch",
    "actor_rollout_ref.actor.optim.lr": "learning rate",
    "actor_rollout_ref.actor.clip_ratio_low": "ε_low",
    "actor_rollout_ref.actor.clip_ratio_high": "ε_high",
    "actor_rollout_ref.actor.policy_loss.loss_mode": "policy loss (Agent Lightning's)",
    "actor_rollout_ref.actor.use_kl_loss": "KL to reference model in the loss",
    "actor_rollout_ref.actor.entropy_coeff": "entropy bonus",
    "actor_rollout_ref.actor.fsdp_config.param_offload": "offload params to CPU when idle",
    "algorithm.adv_estimator": "advantage estimator",
    "algorithm.norm_adv_by_std_in_grpo": "divide by group std (False = Dr. GRPO)",
    "algorithm.enable_rollout_level_advantage": "Agent Lightning: advantage per rollout, not per row",
    "agentlightning.trace_aggregator.level": "Agent Lightning: trajectory or transition",
    "agentlightning.local.agent_class": "Agent Lightning: agent launched by the Controller",
    "agentlightning.rollout_timeout_seconds": "Agent Lightning: per-rollout timeout",
    "trainer.total_epochs": "epochs over the training set",
    "trainer.test_freq": "validate every N steps",
}
for key, meaning in KEYS.items():
    print(f"{key:58} = {str(OmegaConf.select(config, key)):40} # {meaning}")

In [ ]:
bs, n, mbs = config.data.train_batch_size, config.actor_rollout_ref.rollout.n, config.actor_rollout_ref.actor.ppo_mini_batch_size
micro = config.actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu
print(f"""Batch-size arithmetic for one Calc-X training step:
  {bs} questions × n={n}               = {bs * n} rollouts
  each rollout → ≥1 rows (trace aggregation), say R rows in total
  mini-batch = ppo_mini_batch_size × n = {mbs} × {n} = {mbs * n} rows   → R is trimmed to a multiple of {mbs * n}
  optimizer steps per training step    = ppo_epochs × R / {mbs * n}
  each mini-batch = {mbs * n} / {micro} = {mbs * n // micro} micro-batches of {micro} rows (gradient accumulation)""")

## 8. Launch a short real training run

Time to let the real trainer do everything automatically. We reuse the Gateway and Controller that are already running, and stop our standalone vLLM, because verl starts its own engine inside Ray and needs the GPU memory.

This runs `examples/calc_x/train_calc_agent.py` (the same entrypoint `run_local.sh` uses) with overrides that keep it short:

* 3 training steps of 8 questions × 4 rollouts each;
* validation on the 20-question `test_mini.parquet` before training and after the last step;
* no checkpoint saving; metrics go to the console and a JSONL file (no W&B).

Expect about 2-5 minutes. Most of it is one-time startup: Ray, loading the FSDP actor, and starting vLLM with CUDA graphs. Selected progress lines are printed below, and the full log is in `/tmp/agl-tutorial/trainer.log`. In the full log we'll also see:
* `ERROR … hermes_tool_parser … JSONDecodeError` lines from vLLM. These mean the model wrote a malformed `<tool_call>` JSON. That's normal for a small model, the agent just gets plain text back, and it's one of the behaviors RL can train away.
* An "`Exception ignored in … DataLoader.__del__ … killed by signal`" traceback at shutdown. It's harmless dataloader teardown noise.

In [ ]:
stop_service("vllm")
time.sleep(5)
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free / 2**30:.1f} / {total / 2**30:.1f} GiB")

In [ ]:
EXPERIMENT = "calc_x_tutorial"
METRICS_DIR = LOG_DIR / "metrics"
train_cmd = [
    sys.executable, "train_calc_agent.py",
    "--agl-base-url", AGL_URL, "--agl-key", AGL_KEY, "--model", MODEL,
    "--train-file", "data/train.parquet", "--val-file", "data/test_mini.parquet", "--run-name", "tutorial",
    "trainer.total_training_steps=3",
    "data.train_batch_size=8",
    "actor_rollout_ref.actor.ppo_mini_batch_size=8",
    "trainer.val_before_train=True",
    "trainer.save_freq=-1",
    "trainer.logger=[console,file]",
]
trainer_proc = start_service("trainer", train_cmd, cwd=REPO / "examples/calc_x", env={
    "WANDB_MODE": "disabled",
    "VERL_FILE_LOGGER_ROOT": str(METRICS_DIR),
})

INTERESTING = re.compile(r"Initial validation|Final validation|Training Progress|AglRolloutManager: completed=(\d+)/\1 |step:\d+ ")
t0 = time.time()
with open(LOG_DIR / "trainer.log", errors="replace") as log:
    while True:
        line = log.readline()
        if not line:
            if trainer_proc.poll() is not None:
                break
            time.sleep(1)
            continue
        if INTERESTING.search(line) and "vLLMHttpServer" not in line:
            line = re.sub(r"\x1b\[[0-9;]*m", "", line).strip()
            print(f"[{time.time() - t0:5.0f}s] {line[:200]}{' …' if len(line) > 200 else ''}")

print(f"\ntrainer exited with code {trainer_proc.returncode} after {time.time() - t0:.0f}s")
if trainer_proc.returncode != 0:
    raise RuntimeError(f"training failed, last log lines:\n{log_tail('trainer', 40)}")

### 8.1 Reading the metrics

verl logs a few hundred metrics per step. These are the ones to look at first:

| Metric | What it tells us |
|---|---|
| `training/reward` | mean reward of this step's rollouts (noisy with small batches) |
| `val/reward` | mean reward on the validation set (greedy decoding); the number we actually care about |
| `training/n_zero_adv_groups` / `training/n_groups` | how many groups carried no learning signal (§6.3) |
| `training/n_sample_trained`, `training/n_sample_dropped/same_reward` | rows that went into the update, and rows dropped (same-reward groups first) to align with mini-batches |
| `actor/pg_loss`, `actor/grad_norm` | the update itself (notebook 1, §1.7) |
| `actor/ppo_kl`, `actor/pg_clipfrac` | how far the policy moved within a step (notebook 1, §1.7); should stay small |
| `actor/entropy_loss` | mean token entropy of the policy (logged even though `entropy_coeff = 0`); collapsing entropy means less exploration |
| `rollout_corr/kl` | training-vs-inference log-prob mismatch (notebook 1, §2.6) |
| `response_length/mean`, `training/n_turns` | how the agent's behavior is changing |
| `timing_s/gen`, `timing_s/old_log_prob`, `timing_s/update_actor`, `timing_s/update_weights` | where the time goes (rollouts, old log-probs, update, weight sync) |

In [ ]:
metrics_file = METRICS_DIR / "agentlightning" / f"{EXPERIMENT}.jsonl"
logged = [json.loads(line) for line in metrics_file.read_text().splitlines()]
COLUMNS = [
    "val/reward", "training/reward", "training/n_groups", "training/n_zero_adv_groups",
    "training/n_sample_collected", "training/n_sample_dropped/same_reward", "training/n_sample_trained",
    "actor/pg_loss", "actor/grad_norm", "actor/ppo_kl", "actor/pg_clipfrac", "actor/entropy_loss", "rollout_corr/kl",
    "response_length/mean", "timing_s/gen", "timing_s/old_log_prob", "timing_s/update_actor", "timing_s/update_weights", "timing_s/step",
]
metrics_df = pd.DataFrame([{"step": rec["step"], **{c: rec["data"].get(c) for c in COLUMNS}} for rec in logged]).groupby("step").last()
metrics_df.T

Step 0 holds the validation run before training. Three steps on 8 questions won't move `val/reward` meaningfully: 20 validation questions is very noisy, and a few hundred rollouts is a tiny amount of RL. With only 8 groups per step, it's also common for a step to have *all* groups with zero advantage. We'll then see `grad_norm = 0` and every row in `n_sample_dropped/same_reward`: §6.3 in action. What matters is that we can now read every line of this table and know which piece of the pipeline produced it.

The full Calc-X recipe (32 questions per step, 2 epochs over 8,192 questions = 512 steps) takes a few hours on one GPU. Run it from a terminal at the repo root:

```bash
./run_calc_x_local.sh                                   # metrics to console + examples/calc_x/metrics_log + tensorboard
# or, with Weights & Biases:
source .venv/bin/activate && cd examples/calc_x && ./run_local.sh
```

Note that `run_local.sh` uses Gateway port 8181 and cleans up with `pkill -f agl-server` / `ray stop --force`. Make sure nothing else is using port 8181, and stop this notebook's services first (next cell).

## 9. Clean up

Stop everything this notebook started (Controller, Gateway, and vLLM if it's still running).

In [ ]:
stop_service("vllm")

In [ ]:
stop_all()
agl.close()
print("remaining services:", list(SERVICES))

## 10. Where to go from here

### Reading
* Repo docs: [Basics](https://microsoft.github.io/agent-lightning/stable/05-basics/) · [Trainer configuration](https://microsoft.github.io/agent-lightning/stable/20-trainer-configuration/) · [API Gateway](https://microsoft.github.io/agent-lightning/stable/25-api-gateway-configuration/) · [Controller](https://microsoft.github.io/agent-lightning/stable/30-controller-configuration/) · [Asynchronous training](https://microsoft.github.io/agent-lightning/stable/35-asynchronous-training/) · [Calc-X example](https://microsoft.github.io/agent-lightning/stable/50-example-calc-x/)
* Code, in reading order (`agentlightning/verl/` in the installed package, or [on GitHub](https://github.com/microsoft/agent-lightning/blob/v1.0.2/agentlightning/verl)): `trainer.py` (`_train_step`, `fit`) → `agl_rollout_manager.py` → `rollout_adapter.py` → `rollout_level_advantage.py` → `per_rollout_loss.py`
* verl internals: `.venv/lib/python3.12/site-packages/verl/trainer/ppo/ray_trainer.py` (`RayPPOTrainer`), `trainer/ppo/core_algos.py` (advantage estimators and policy losses), `workers/utils/losses.py` (`ppo_loss`)
* Papers: PPO (Schulman et al., 2017) · GRPO (DeepSeekMath, Shao et al., 2024) · DAPO (clip-higher, 2025) · Dr. GRPO (Liu et al., 2025) · HybridFlow, the verl paper (Sheng et al., 2024)
* For step-through debugging of the same rollout path, see `debug/rollout_step.py` and the "AGL: Sandbox" launch configuration in `.vscode/launch.json`.